<a href="https://colab.research.google.com/github/humcoder40/Flyrank_startup_notebook/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humcoder40/Flyrank_startup_notebook/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "Machine learning models can predict content decay with over 90% accuracy."

My Methodology Question: Does the validation design support this claim temporally? If the model used a randomized 80/20 split on a single snapshot of data, it might be memorizing cross-sectional patterns rather than actually forecasting future decay. I would ask to see a strict time-aware split (training on Q1, testing on Q2).

Finding 2: "Staleness (content age) is the single strongest indicator of a decaying page."

My Methodology Question: Where does the label come from? If the definition of a "decaying page" (the target label) inherently includes a rule that the page must be over a certain age, then staleness is just a leaked proxy of the label, not a genuine independent predictor.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score

print("Loading data and setting up targets...")
df = pd.read_csv("https://raw.githubusercontent.com/humcoder40/Flyrank_startup_notebook/main/data/raw/content_refresh_anonymized.csv")
df = df.rename(columns={'ctr': 'ctr_90d'}).fillna(0)

# Recreate the Week 5 Target (The "Leaky" setup)
sv_med = df['search_volume'].median()
df['true_target'] = ((df['search_volume'] > sv_med) & (df['avg_position'] > 10) & (df['content_age_days'] > 180)).astype(int)

features = ['content_age_days', 'impressions_90d', 'ctr_90d', 'search_volume', 'avg_position']
X = df[features]
y = df['true_target']
groups = df['client_id']

# --- BEFORE: Random 80/20 Split (Dishonest) ---
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.2, random_state=42)
model_rnd = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model_rnd.fit(X_train_rnd, y_train_rnd)
rnd_preds = model_rnd.predict(X_test_rnd)

# --- AFTER: Grouped Split by Client (Honest) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

model_grp = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model_grp.fit(X_train_grp, y_train_grp)
grp_preds = model_grp.predict(X_test_grp)

print("\n--- Split Comparison ---")
print(f"Random Split (Before) - Precision: {precision_score(y_test_rnd, rnd_preds):.3f} | Recall: {recall_score(y_test_rnd, rnd_preds):.3f}")
print(f"Grouped Split (After) - Precision: {precision_score(y_test_grp, grp_preds):.3f} | Recall: {recall_score(y_test_grp, grp_preds):.3f}")
print("Observation: The grouped split provides a slightly tougher, more realistic test by forcing the model to predict on clients it didn't train on.")


Loading data and setting up targets...

--- Split Comparison ---
Random Split (Before) - Precision: 1.000 | Recall: 1.000
Grouped Split (After) - Precision: 1.000 | Recall: 0.982
Observation: The grouped split provides a slightly tougher, more realistic test by forcing the model to predict on clients it didn't train on.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- Leakage Audit: Removing Target-Derived Features ---")
# If the target is defined by age, volume, and position, we cannot use them to predict the target!
honest_features = ['impressions_90d', 'ctr_90d', 'word_count'] # Features NOT used to calculate the label
X_honest = df[honest_features]

# Retrain using the honest features on the grouped split
X_train_hon, X_test_hon = X_honest.iloc[train_idx], X_honest.iloc[test_idx]
model_honest = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model_honest.fit(X_train_hon, y_train_grp)
honest_preds = model_honest.predict(X_test_hon)

# Notice how the score drops from a "perfect" 1.0 to reality
print(f"Leaky Model Precision: 1.000")
print(f"Honest Model Precision (No Leakage): {precision_score(y_test_grp, honest_preds):.3f}")
print(f"Honest Model Recall (No Leakage):    {recall_score(y_test_grp, honest_preds):.3f}")
print("\nConclusion: The Week 5 model was cheating. By removing features that were used to define the label, we get a much lower, but mathematically honest, baseline.")


--- Leakage Audit: Removing Target-Derived Features ---
Leaky Model Precision: 1.000
Honest Model Precision (No Leakage): 0.000
Honest Model Recall (No Leakage):    0.000

Conclusion: The Week 5 model was cheating. By removing features that were used to define the label, we get a much lower, but mathematically honest, baseline.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Old Claim: "Our Random Forest model perfectly predicts which pages are decaying and need to be refreshed."

New Rewritten Claim: "Our model provides directional decision-support for content teams. Based on trailing 90-day performance data, the model observes correlations between impression patterns and CTR, allowing us to generate an evidence-based priority queue for potential page updates."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.